# Convolutional Neural Networks: Architecture Design on CIFAR-10

**AREP Assignment — Convolutional Layers as Inductive Bias**

This notebook is not a tutorial to follow. It is an experiment: we choose a dataset, build a non-convolutional baseline, design a convolutional architecture from scratch, run a controlled experiment on one architectural choice, and interpret the results.

The guiding question throughout is not *"does this code run?"* but *"why does this architectural choice make sense for this data?"*

## Learning Objectives

By the end of this notebook, we should be able to:

- Understand the role and mathematical intuition behind convolutional layers.
- Analyze how architectural decisions (kernel size, depth, stride, padding) affect learning.
- Compare convolutional layers with fully connected layers for image-like data.
- Perform a minimal but meaningful exploratory data analysis (EDA).
- Communicate architectural and experimental decisions clearly.

## 0. Setup

In [ ]:
# NumPy handles arrays and vectorized numerical operations.
import numpy as np

# Matplotlib is used to visualize images, class distributions, and training curves.
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
    from tensorflow.keras.losses import SparseCategoricalCrossentropy
    from tensorflow.keras.optimizers import Adam

    TF_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except Exception as exc:
    TF_AVAILABLE = False
    print("TensorFlow is not available in this environment.")
    print("Reason:", repr(exc))

## 1. Dataset Selection and Justification

**Dataset: CIFAR-10**

CIFAR-10 contains 60,000 color images (32×32×3), evenly split across 10 mutually exclusive classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck): 50,000 for training and 10,000 for testing.

**Why CIFAR-10 is appropriate for convolutional layers, and not just for dense layers**

The previous session's MNIST notebook used dense layers on flattened `28x28` grayscale digits and worked reasonably well, because digit shapes are simple, high-contrast, and centered. CIFAR-10 is a deliberately harder case for a dense network, which is exactly why it is a good test bed for convolution:

- **Color channels.** Each image has 3 channels (RGB), so a flattened vector has `32 * 32 * 3 = 3072` inputs. A dense layer connected to all of them has no notion that channel values at the same pixel location belong together.
- **Spatial variability.** Objects (a cat, a truck) appear at different positions, scales, and orientations within the frame. A dense layer learns a separate weight for every pixel position, so a pattern learned in one corner of the image does not transfer to the same pattern appearing elsewhere. A convolutional filter, by contrast, is applied across all spatial positions, so it can detect the same local pattern (an edge, a texture, a corner) regardless of where it appears.
- **Local structure matters more than raw pixel identity.** What makes an image "cat-like" is local structure (fur texture, ear shapes, eye patterns) composed hierarchically into larger patterns. Convolution's inductive bias — that nearby pixels are related and that the same local feature detector is useful everywhere in the image — matches this structure directly. A dense layer has to learn this from data with no structural help, which requires far more parameters and more data to generalize.

This makes CIFAR-10 a good dataset to demonstrate *why* convolution exists, not just that it happens to perform better.

## 2. Exploratory Data Analysis (EDA)

Before designing any architecture, we look at the data directly: how many examples, how many classes, what the images look like, and what preprocessing they need.

In [ ]:
if TF_AVAILABLE:
    # load_data returns a standard split: 50,000 training examples, 10,000 test examples.
    (X_train_raw, y_train), (X_test_raw, y_test) = tf.keras.datasets.cifar10.load_data()

    # Labels arrive as (N, 1) column vectors; flatten to (N,) for convenience.
    y_train = y_train.reshape(-1)
    y_test = y_test.reshape(-1)

    class_names = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

    print("X_train_raw shape:", X_train_raw.shape)
    print("y_train shape:", y_train.shape)
    print("X_test_raw shape:", X_test_raw.shape)
    print("y_test shape:", y_test.shape)
    print("Pixel value range:", X_train_raw.min(), "to", X_train_raw.max())
    print("Number of classes:", len(class_names))
else:
    print("Skipping CIFAR-10 loading because TensorFlow is not available.")

### Class Distribution

CIFAR-10 is a balanced dataset by construction (6,000 images per class), but we verify this rather than assume it — an unverified assumption about class balance is exactly the kind of thing that silently breaks a metrics discussion later.

In [ ]:
if TF_AVAILABLE:
    train_counts = np.bincount(y_train, minlength=10)
    test_counts = np.bincount(y_test, minlength=10)

    x_positions = np.arange(10)
    width = 0.35

    plt.figure(figsize=(9, 4))
    plt.bar(x_positions - width / 2, train_counts, width, label="train")
    plt.bar(x_positions + width / 2, test_counts, width, label="test")
    plt.xticks(x_positions, class_names, rotation=45, ha="right")
    plt.ylabel("count")
    plt.title("CIFAR-10 class distribution")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Train counts per class:", train_counts)
    print("Test counts per class:", test_counts)
else:
    print("Skipping class distribution because TensorFlow is not available.")

### Sample Images per Class

Always look at the actual images before modeling. This is also where you notice things a table of numbers hides: low resolution, ambiguous poses, background clutter.

In [ ]:
if TF_AVAILABLE:
    plt.figure(figsize=(10, 4))
    for class_index in range(10):
        # Show the first example found for each class.
        example_index = np.where(y_train == class_index)[0][0]
        plt.subplot(2, 5, class_index + 1)
        plt.imshow(X_train_raw[example_index])
        plt.title(class_names[class_index])
        plt.axis("off")
    plt.suptitle("One example per class")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping sample visualization because TensorFlow is not available.")

### Preprocessing Needed

From the EDA above, two things need to happen before this data can be used for training:

1. **Normalization.** Pixel values are integers in `[0, 255]`. We scale them to `[0, 1]` by dividing by 255, which makes optimization easier (same reasoning as the MNIST notebook).
2. **Shape handling depends on the model, not the data.** Unlike the MNIST dense-network notebook, we do **not** flatten images by default here. A `Conv2D` layer expects a `(height, width, channels)` tensor — flattening would destroy the spatial structure convolution is meant to exploit. We keep images as `(32, 32, 3)` for the CNN, and only flatten separately for the dense baseline in Section 3.

No resizing is needed: all CIFAR-10 images are already a uniform `32x32x3`.

In [ ]:
if TF_AVAILABLE:
    # Convert integer pixels from 0-255 into floats from 0-1.
    X_train = X_train_raw.astype("float32") / 255.0
    X_test = X_test_raw.astype("float32") / 255.0

    # Keep the spatial shape (32, 32, 3) for the CNN.
    print("X_train shape (kept for CNN):", X_train.shape)

    # Flattened version, kept only for the dense baseline in Section 3.
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)
    print("X_train_flat shape (for baseline):", X_train_flat.shape)
else:
    print("Skipping preprocessing because TensorFlow is not available.")

## 3. Baseline Model (Non-Convolutional)

Before building a CNN, we establish a reference point: a dense network with no convolutional layers, trained on the flattened images (`3072` inputs). Its purpose is not to be a good model — it is to give us a number and a set of failure modes to compare the CNN against in Section 6.

**Architecture choice.** We reuse the same shape of network as the MNIST notebook (`784 -> 25 -> 15 -> 10` there), scaled to CIFAR-10's larger, 3-channel input: `3072 -> 128 -> 64 -> 10`. This keeps the comparison fair — similar depth and a similar "recipe" (ReLU hidden layers, linear output logits, Adam, sparse categorical crossentropy) — so that whatever performance gap we see later is attributable to convolution, not to an arbitrarily different training setup.

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(7)

    baseline_model = Sequential([
        Dense(units=128, activation="relu", input_shape=(3072,)),
        Dense(units=64, activation="relu"),
        Dense(units=10, activation="linear"),
    ], name="dense_baseline")

    baseline_model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    baseline_model.summary()
else:
    print("Skipping baseline model creation because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    baseline_history = baseline_model.fit(
        X_train_flat,
        y_train,
        validation_split=0.1,
        epochs=15,
        batch_size=128,
        verbose=1,
    )
else:
    print("Skipping baseline training because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(baseline_history.history["loss"], label="train")
    plt.plot(baseline_history.history["val_loss"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Baseline: training and validation loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(baseline_history.history["accuracy"], label="train")
    plt.plot(baseline_history.history["val_accuracy"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Baseline: training and validation accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(X_test_flat, y_test, verbose=0)
    baseline_param_count = baseline_model.count_params()

    print("Baseline test loss:", baseline_test_loss)
    print("Baseline test accuracy:", baseline_test_accuracy)
    print("Baseline parameter count:", baseline_param_count)
else:
    print("Skipping baseline evaluation because TensorFlow is not available.")

### Baseline Summary and Observed Limitations

*(Fill in the printed numbers above once you run this notebook — they will differ slightly by machine/seed behavior, so report your actual run.)*

**Architecture:** `Flatten(3072) -> Dense(128, relu) -> Dense(64, relu) -> Dense(10, linear)`

**Expected limitations of this baseline, given what Section 1 argued:**

- **No translation invariance.** Every one of the 3072 input weights connects to a fixed pixel position. If the network learns to recognize a wheel shape in one part of the image, that knowledge does not transfer to the same shape appearing elsewhere — it has to relearn it, which wastes capacity and data.
- **No explicit locality prior.** Dense layers treat all 3072 inputs as equally likely to interact. There is nothing in the architecture that tells the model "nearby pixels are more likely to be related than distant ones" — it has to discover this from data alone, which is harder with only 50,000 training examples.
- **Parameter inefficiency relative to the problem.** Most of the baseline's parameters live in the first dense layer (`3072 * 128` weights). A large share of that capacity is spent just accounting for *where* a pattern occurs, rather than *what* the pattern is.
- **Expected gap between train and validation accuracy.** Because the model has many parameters relative to the structure it can exploit, it is more prone to memorizing pixel-position-specific patterns from the training set that do not generalize — watch for training accuracy pulling ahead of validation accuracy in the curves above.

This is the reference point Section 6 will compare the CNN against.

## 4. Convolutional Architecture Design

We design a small CNN, intentionally simple, and justify every choice rather than copying a known architecture wholesale.

**Overall shape:**

```text
Input (32,32,3)
  -> Conv2D(32 filters, 3x3, stride 1, padding "same", relu)
  -> MaxPooling2D(2x2)
  -> Conv2D(64 filters, 3x3, stride 1, padding "same", relu)
  -> MaxPooling2D(2x2)
  -> Flatten
  -> Dense(64, relu)
  -> Dense(10, linear)   # logits, same convention as the baseline and the MNIST notebook
```

**Justification for each choice:**

- **Number of convolutional layers (2, in 2 blocks).** One conv layer only sees raw pixel-level patterns (edges, color blobs). A second conv layer, operating on the feature maps produced by the first, can combine those into larger patterns (textures, simple shapes). Two blocks is the minimum depth that lets the network build a real feature *hierarchy* instead of a single filter bank — while staying small enough to train quickly and to reason about by hand, which matters more here than squeezing out extra accuracy.
- **Kernel size (3x3 throughout).** A 3x3 kernel is the smallest size that still captures 2D directional structure (an edge at an angle, a corner) rather than just a single pixel's neighborhood. It is also the standard choice precisely because stacking two 3x3 convolutions covers a 5x5 receptive field with fewer parameters (`2 * 3*3 = 18` weights per input-output channel pair) than one 5x5 convolution (`25` weights), while adding an extra ReLU non-linearity in between.
- **Stride (1) and padding ("same").** We keep stride 1 with "same" padding inside the conv layers so that *downsampling is a deliberate, separate decision* (done by pooling) rather than an incidental side effect of the convolution's stride. This keeps the two mechanisms — feature extraction (conv) and spatial reduction (pooling) — conceptually and architecturally separate, which makes the model easier to reason about and to modify in the controlled experiment in Section 5.
- **Activation (ReLU) after each conv layer.** Same choice as the baseline and the MNIST notebook, for a fair comparison and for the same reason: ReLU is cheap to compute and avoids the vanishing-gradient problem that sigmoid/tanh have for deeper stacks.
- **Pooling (2x2 max pooling after each conv block).** Max pooling does two things here: it reduces the spatial resolution by half at each stage (`32x32 -> 16x16 -> 8x8`), which controls the parameter count of the eventual `Flatten` + `Dense` layers, and it adds a small amount of local translation invariance — a feature detected in slightly different positions within a 2x2 window still produces the same pooled output. We use *max* rather than average pooling because for detecting "is this pattern present in this region," the strongest activation is usually more informative than the regional average.
- **Increasing filter count with depth (32 -> 64).** As spatial resolution shrinks (fewer pixels per feature map), we increase the number of filters (more channels), following the common trade-off of exchanging spatial resolution for representational depth: early layers need few filters to detect simple, generic patterns (edges, colors), while later layers benefit from more filters to represent a larger vocabulary of more complex, combined patterns.
- **Final layers (`Flatten -> Dense(64) -> Dense(10)`).** Once convolution and pooling have extracted a compact, spatially-reduced feature representation (`8*8*64 = 4096` values), a small dense head is enough to combine those features into class scores. This mirrors the baseline's final dense layers, again to keep the comparison in Section 6 about convolution itself, not about differences in the classifier head.

In [ ]:
def build_cnn(filters=(32, 64), kernel_size=3, pool_size=2, dense_units=64, name="cnn"):
    # Small, intentional CNN: two conv+pool blocks, then a dense head.
    # Kept as a function so Section 5 can reuse it with different hyperparameters
    # while keeping everything else identical (a controlled experiment).
    return Sequential([
        Conv2D(filters[0], kernel_size, strides=1, padding="same",
               activation="relu", input_shape=(32, 32, 3)),
        MaxPooling2D(pool_size),
        Conv2D(filters[1], kernel_size, strides=1, padding="same", activation="relu"),
        MaxPooling2D(pool_size),
        Flatten(),
        Dense(dense_units, activation="relu"),
        Dense(10, activation="linear"),
    ], name=name)

if TF_AVAILABLE:
    tf.random.set_seed(7)

    cnn_model = build_cnn(name="cnn_baseline_design")

    cnn_model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    cnn_model.summary()
else:
    print("Skipping CNN model creation because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    cnn_history = cnn_model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=15,
        batch_size=128,
        verbose=1,
    )
else:
    print("Skipping CNN training because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(cnn_history.history["loss"], label="train")
    plt.plot(cnn_history.history["val_loss"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("CNN: training and validation loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(cnn_history.history["accuracy"], label="train")
    plt.plot(cnn_history.history["val_accuracy"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("CNN: training and validation accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(X_test, y_test, verbose=0)
    cnn_param_count = cnn_model.count_params()

    print("CNN test loss:", cnn_test_loss)
    print("CNN test accuracy:", cnn_test_accuracy)
    print("CNN parameter count:", cnn_param_count)

    print("\nBaseline vs CNN")
    print(f"{'':20s} {'test acc':>10s} {'params':>12s}")
    print(f"{'dense baseline':20s} {baseline_test_accuracy:10.3f} {baseline_param_count:12d}")
    print(f"{'cnn':20s} {cnn_test_accuracy:10.3f} {cnn_param_count:12d}")
else:
    print("Skipping CNN evaluation because TensorFlow is not available.")

## 5. Controlled Experiment: Kernel Size (3x3 vs 5x5)

**Variable under test:** convolutional kernel size.

**Everything else held fixed:** number of conv layers (2), filters per layer (32, 64), stride (1), padding ("same"), activation (ReLU), pooling (2x2 max pooling), dense head (64 units), optimizer (Adam, `lr=1e-3`), loss, epochs (15), batch size (128), and the random seed.

**Why this variable.** Section 4 justified 3x3 kernels partly by argument (two stacked 3x3 convolutions match a 5x5 receptive field with fewer parameters and an extra non-linearity). This experiment tests that argument empirically: does a single-layer-equivalent 5x5 kernel actually perform differently from 3x3 in this architecture, and at what parameter cost?

**Prediction before running:** the 5x5 variant will have noticeably more parameters per conv layer (`5*5=25` vs `3*3=9` weights per input-output channel pair, ~2.8x more), and, following the reasoning in Section 4, we expect it to not clearly outperform the 3x3 model — if it doesn't, that is evidence in favor of preferring the smaller kernel for a similar receptive field at lower cost.

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(7)

    cnn_model_5x5 = build_cnn(kernel_size=5, name="cnn_kernel5x5")

    cnn_model_5x5.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    cnn_model_5x5.summary()
else:
    print("Skipping 5x5 CNN creation because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    cnn_5x5_history = cnn_model_5x5.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=15,
        batch_size=128,
        verbose=1,
    )
else:
    print("Skipping 5x5 CNN training because TensorFlow is not available.")

In [ ]:
if TF_AVAILABLE:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(cnn_history.history["val_loss"], label="3x3 (val)")
    plt.plot(cnn_5x5_history.history["val_loss"], label="5x5 (val)")
    plt.xlabel("Epoch")
    plt.ylabel("Validation loss")
    plt.title("Validation loss: kernel size")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(cnn_history.history["val_accuracy"], label="3x3 (val)")
    plt.plot(cnn_5x5_history.history["val_accuracy"], label="5x5 (val)")
    plt.xlabel("Epoch")
    plt.ylabel("Validation accuracy")
    plt.title("Validation accuracy: kernel size")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    cnn_5x5_test_loss, cnn_5x5_test_accuracy = cnn_model_5x5.evaluate(X_test, y_test, verbose=0)
    cnn_5x5_param_count = cnn_model_5x5.count_params()

    print("Kernel size comparison")
    print(f"{'':16s} {'test acc':>10s} {'test loss':>10s} {'params':>12s}")
    print(f"{'3x3':16s} {cnn_test_accuracy:10.3f} {cnn_test_loss:10.3f} {cnn_param_count:12d}")
    print(f"{'5x5':16s} {cnn_5x5_test_accuracy:10.3f} {cnn_5x5_test_loss:10.3f} {cnn_5x5_param_count:12d}")
    print(f"\nParameter increase (5x5 vs 3x3): {cnn_5x5_param_count / cnn_param_count:.2f}x")
else:
    print("Skipping kernel size comparison because TensorFlow is not available.")

### Experiment Summary: Quantitative, Qualitative, and Trade-offs

*(Complete this after running the cells above with your actual numbers — the placeholders below describe what to look for and how to reason about it, not the answer.)*

**Quantitative results.** Report the test accuracy, test loss, and parameter count for both kernels from the table above, and the parameter multiplier (5x5 has `25/9 ≈ 2.78x` as many weights per conv filter as 3x3).

**Qualitative observations to check for:**

- Did the 5x5 model converge faster or slower per epoch (look at the validation curves)? A larger kernel sees more context per layer, which can help early on, but also means more parameters to fit.
- Did the 5x5 model show a larger gap between training and validation accuracy than 3x3? More parameters with the same amount of data increases the risk of overfitting.
- Was the final test accuracy meaningfully different, or within noise of a single run (a few tenths of a percentage point)?

**Trade-off (performance vs complexity).** If 5x5 does not clearly outperform 3x3 despite costing ~2.8x more parameters per conv layer, that supports the standard practice (and the Section 4 justification) of preferring smaller kernels stacked in depth over larger kernels: similar or better effective receptive field, fewer parameters, lower risk of overfitting, and one extra non-linearity for the same receptive field when using two 3x3 layers instead of one 5x5 layer. If 5x5 clearly outperforms 3x3 here, that is also worth reporting honestly — it would suggest this particular dataset/architecture benefits from a larger single-layer receptive field, which is itself an interesting, reportable finding.

## 6. Interpretation and Architectural Reasoning

*(This section is graded heavily on reasoning, not on restating numbers. Fill in the bracketed `[...]` with your actual results from Sections 3–5 before submitting, and adjust the reasoning if your results disagree with the expectations below — an honest account of a surprising result is worth more than a tidy one that doesn't match your data.)*

### Why did convolutional layers outperform (or not) the baseline?

The dense baseline (Section 3) reached `[baseline_test_accuracy]` test accuracy with `[baseline_param_count]` parameters. The CNN (Section 4) reached `[cnn_test_accuracy]` with `[cnn_param_count]` parameters.

If the CNN outperformed the baseline, the mechanism is not "more parameters" — compare the two parameter counts directly. The likely explanation is that convolution gives the network a *head start* that matches the true structure of image data: a filter that learns to detect an edge or a color transition in one region of the image is automatically reused everywhere in the image, because the same kernel weights slide across all spatial positions. The dense baseline has to independently discover and store a version of that same pattern-detector for every pixel position where it might appear, which is a much harder learning problem given the same 50,000 training images. This is directly visible in the training curves: the baseline is more likely to show training accuracy pulling further ahead of validation accuracy than the CNN does, because it has more "position-specific" ways to overfit the training set.

If the CNN did *not* clearly outperform the baseline (or the gap was smaller than expected), plausible explanations to consider: only 15 epochs of training, no data augmentation, a small/simple CNN (2 conv blocks), or CIFAR-10's low resolution (32x32) limiting how much local structure convolution can exploit before pooling discards it. Report which of these is most likely given what you observed in Sections 3–5.

### What inductive bias does convolution introduce?

An inductive bias is an assumption built into the architecture, before seeing any data, about what kinds of patterns are worth looking for. Convolution encodes two specific assumptions:

1. **Locality.** Nearby pixels are assumed to be more relevant to each other than distant ones — a convolutional filter only ever looks at a small neighborhood (e.g., 3x3) at a time, never at the whole image at once.
2. **Translation equivariance / weight sharing.** The same filter (the same set of weights) is applied at every spatial position. This encodes the assumption that a useful pattern (an edge, a texture) is equally worth detecting no matter where in the image it appears, so the same detector should be reused rather than relearned per position.

These are *assumptions*, not guarantees — they are what make convolution efficient and effective specifically when the assumption holds, which for natural images it usually does.

### In what type of problems would convolution not be appropriate?

Convolution's locality and translation-equivariance assumptions are a poor fit when the data does not have that kind of spatial/local structure:

- **Tabular data without spatial or sequential meaning** (e.g., a table of customer attributes: age, income, tenure). There is no meaningful notion of "neighboring columns" — column order is often arbitrary, so a filter sliding across columns would be learning from a relationship that doesn't exist.
- **Data where global, not local, structure dominates and position is meaningful, not something to be invariant to** — for example, if the exact position of a feature in a fixed-format input is itself the signal (some highly structured sensor layouts or forms), translation equivariance actively throws away useful information.
- **Very small or already globally-summarized feature vectors**, where there is nothing "local" left to exploit — convolution's main advantage (parameter sharing across many spatial positions) disappears if there are only a handful of input features to begin with.

In those cases, dense layers (which make no locality assumption) or architectures matched to the data's actual structure (e.g., graph neural networks for graph-structured data, recurrent/attention-based models for sequences where order matters but "locality" is looser) are usually a better fit than convolution.

## 7. Deployment in SageMaker

This section trains the chosen CNN (the 3x3 design from Section 4) as a SageMaker **training job**, then deploys it to a real-time SageMaker **endpoint**.

**Run this section inside SageMaker Studio**, not locally: Studio comes with the execution role and AWS region already configured, so there is no AWS CLI setup needed. Steps:

1. Open **SageMaker AI → Studio** in the AWS console (create a domain/user profile via "Quick setup" if you don't have one yet).
2. Launch Studio, upload `cnn_cifar10_workshop.ipynb` and `train.py` into the **same folder** there (`train.py` is required — it's the script this section trains).
3. Pick a Python 3 / Data Science kernel and run the notebook top to bottom.

**Important — this section costs money and touches your AWS account.** Training jobs and, especially, live endpoints are billed for as long as they run. Run these cells deliberately, and:

- run the cleanup cell at the end of this section (`predictor.delete_endpoint()`) once you're done testing, and
- stop the Studio app/instance from the "Running Instances" panel when you're finished — Studio itself is also billed while running, separately from the endpoint.

**Why a separate script.** SageMaker's TensorFlow "script mode" trains by running a plain Python entry-point script (`train.py`) inside a managed training container — it does not execute notebook cells directly. `train.py` rebuilds the same `build_cnn` architecture from Section 4 (2 conv+pool blocks, 3x3 kernels) so that what gets deployed is the same design justified and evaluated above, not a different model.

In [ ]:
import sagemaker
from sagemaker.tensorflow import TensorFlow

sagemaker_session = sagemaker.Session()

# Inside SageMaker Studio this resolves automatically to the domain's execution role.
role = sagemaker.get_execution_role()

print("Using role:", role)
print("Default bucket:", sagemaker_session.default_bucket())

### Getting CIFAR-10 into this environment

This Studio domain has no general internet access (confirmed: even `https://www.google.com` times out from a terminal here), so downloading CIFAR-10 directly — from Keras's default URL or any other external mirror — will not work from inside Studio. The fix: get the dataset in through S3 instead, which uses AWS's own network path, not the blocked internet path.

**One-time setup (do this before running the cells below):**

1. **Locally** (where the earlier sections of this notebook already downloaded CIFAR-10 successfully), run:
   ```bash
   python export_cifar10.py
   ```
   This writes `cifar10_data.npz` (~170 MB) next to the notebook.
2. Upload that file via the **S3 console** (not Studio's uploader) to:
   `s3://<your-sagemaker-default-bucket>/cifar10-workshop/cifar10_data.npz`
   (the bucket name is printed by the cell above, e.g. `sagemaker-<region>-<account-id>`).
3. Back in **Studio's terminal**, pull it into this notebook's working directory:
   ```bash
   aws s3 cp s3://<your-sagemaker-default-bucket>/cifar10-workshop/cifar10_data.npz .
   ```

The cell below loads that local file (for testing the endpoint later) and also computes the S3 URI that `estimator.fit()` will use to feed the same file to the training job — SageMaker downloads S3 channel data as a built-in step, independent of whether the training instance itself has general internet access.

In [ ]:
import os

data_path = "cifar10_data.npz"
assert os.path.exists(data_path), (
    f"{data_path} not found in the working directory. "
    "Run export_cifar10.py locally, upload it via the S3 console, "
    "then `aws s3 cp` it here (see steps above) before continuing."
)

cifar_data = np.load(data_path)
X_train_raw, y_train = cifar_data["X_train"], cifar_data["y_train"]
X_test_raw, y_test = cifar_data["X_test"], cifar_data["y_test"]

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

X_train = X_train_raw.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

print("X_train:", X_train.shape, " X_test:", X_test.shape)

# Upload the same local file to S3 so the training job can read it as an input channel.
s3_data_uri = sagemaker_session.upload_data(
    path=data_path,
    bucket=sagemaker_session.default_bucket(),
    key_prefix="cifar10-workshop",
)
print("Training data channel:", s3_data_uri)

In [ ]:
estimator = TensorFlow(
    entry_point="train.py",
    source_dir=".",                      # picks up train.py from this folder
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",        # CPU instance; adjust to a GPU type (e.g. ml.g4dn.xlarge) if you want faster training
    framework_version="2.13",
    py_version="py310",
    hyperparameters={
        "epochs": 15,
        "batch-size": 128,
        "learning-rate": 1e-3,
        "kernel-size": 3,
    },
    base_job_name="cnn-cifar10-workshop",
)

# Feed the S3-staged CIFAR-10 file as the "training" input channel — train.py
# reads it from SM_CHANNEL_TRAINING instead of downloading it from the internet.
estimator.fit({"training": s3_data_uri})

### Deploy to a Real-Time Endpoint

`estimator.deploy(...)` packages the trained model (exported by `train.py` in TensorFlow SavedModel format) into a TensorFlow Serving container and stands up a live HTTPS endpoint that accepts prediction requests.

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

print("Endpoint name:", predictor.endpoint_name)

In [ ]:
if TF_AVAILABLE:
    # Send a handful of real test images to the live endpoint and compare
    # against the true labels, the same way Section 4 evaluated the local model.
    sample_indices = np.arange(5)
    payload = {"instances": X_test[sample_indices].tolist()}

    response = predictor.predict(payload)
    endpoint_logits = np.array(response["predictions"])
    endpoint_predicted = np.argmax(endpoint_logits, axis=1)

    for i, index in enumerate(sample_indices):
        true_name = class_names[y_test[index]]
        pred_name = class_names[endpoint_predicted[i]]
        print(f"true: {true_name:12s}  predicted by endpoint: {pred_name}")
else:
    print("Skipping endpoint test because TensorFlow is not available.")

### Cleanup — Run This When You Are Done

**Do not skip this cell.** A running SageMaker endpoint bills per hour whether or not you send it requests. Delete it as soon as you have captured the results you need for the deliverable (a screenshot or the printed output above is enough evidence for the README).

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted.")

## Summary

| Step | What it established |
|---|---|
| Dataset (Sec. 1) | CIFAR-10 chosen because color + spatial variability make dense layers' lack of locality/translation-invariance a real handicap, not a theoretical one. |
| EDA (Sec. 2) | Balanced classes, `32x32x3` images, only normalization needed as preprocessing. |
| Baseline (Sec. 3) | `Flatten -> Dense -> Dense -> Dense` reference point and its expected failure modes. |
| CNN design (Sec. 4) | 2 conv+pool blocks, 3x3 kernels, stride 1 / "same" padding, ReLU, max pooling, filters growing 32→64 — every choice justified against the baseline's limitations. |
| Experiment (Sec. 5) | 3x3 vs 5x5 kernel, everything else fixed — tests the Section 4 argument empirically. |
| Interpretation (Sec. 6) | Why convolution helped (or didn't), its inductive bias (locality + weight sharing), and where it would be the wrong choice. |
| Deployment (Sec. 7) | The exact Section-4 architecture trained and served through a real SageMaker endpoint, via `train.py`. |

**Before submitting:** fill in every `[...]` placeholder in Sections 3, 5, and 6 with your actual run's numbers, and write the `README.md` (problem description, dataset description, a simple architecture diagram, experimental results, and interpretation) using this notebook as the source of truth.